**1. Installing packages**




In [ ]:
!pip install -q langchain-openai langchain-chroma python-dotenv

**2. Copy API access**

In [ ]:
## DO NOT CHANGE THESE VALUES
LLM_LITE_API_BASE = "https://llmproxy.uva.nl"
LLM_EMBEDDING = "text-embedding-3-large"


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads from .env if present; no-op in Colab

# YOU CAN CHANGE THESE VALUES
LLM_LITE_API_KEY = os.getenv("LLM_API_KEY", "your-api-key-here")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-4.1-nano")  # ['gpt-4o', 'gpt-4.1-nano', 'gpt-4.1', 'gpt-4o-mini', 'gpt-5', 'o3-mini', 'gpt-5.1']

**3. Upload/ Unzip Chroma file**

In [ ]:
!tar -xf chroma-db.tgz

tar: chroma-db.tgz: Cannot open: No such file or directory
tar: Error is not recoverable: exiting now


**4. Load embedding model**

In [ ]:
from langchain_openai.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model=LLM_EMBEDDING,
    openai_api_base=LLM_LITE_API_BASE,
    api_key=LLM_LITE_API_KEY
)

**5. Chroma vector store**

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="cahiers-chunk-08",
    embedding_function=embeddings,
    persist_directory=".",
)


In [ ]:
import chromadb

client = chromadb.PersistentClient(path=".")
collections = client.list_collections()
for col in collections:
  print(col.name)


cahiers-chunk-08


**6. Creating Retriever**

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

**7. Return a list of related article titles**

In [ ]:
def related_titles(docs):
  titles=[]
  for doc in docs:
    metadata=doc.metadata
    article_title=metadata.get("title")
    if article_title not in titles:
      titles.append(article_title)
  return titles

**7.1 Format documents with metadata**

In [ ]:
def format_docs_with_metadata(docs):
  """
  Format documents with their metadata for use in prompts.
  Each document includes title, date, and content.
  """
  formatted_texts = []
  for i, doc in enumerate(docs, 1):
    metadata = doc.metadata
    title = metadata.get("title", "Unknown")
    year = metadata.get("year", "")
    month = metadata.get("month", "")
    day = metadata.get("day", "")

    # Format date if available
    date_str = ""
    if year and month and day:
      date_str = f" (Published: {year}-{month:02d}-{day:02d})"
    elif year:
      date_str = f" (Published: {year})"

    formatted_texts.append(
      f"Document {i}:\n"
      f"Title: {title}{date_str}\n"
      f"Content: {doc.page_content}\n"
    )

  return "\n".join(formatted_texts)

**7.2 Parse date filters from queries**

In [ ]:
import re

def parse_date_filter(query):
  """
  Extract date information from query for metadata filtering.
  Examples:
    - "in 2016" -> {"year": 2016}
    - "in March 2016" -> {"year": 2016, "month": 3}
    - "published in 2015" -> {"year": 2015}
  """
  filter_dict = {}

  # Extract year (4 digits starting with 19 or 20)
  year_match = re.search(r'\b(19|20)\d{2}\b', query)
  if year_match:
    filter_dict['year'] = int(year_match.group())

  # Extract month name
  months = {
    'january': 1, 'february': 2, 'march': 3, 'april': 4,
    'may': 5, 'june': 6, 'july': 7, 'august': 8,
    'september': 9, 'october': 10, 'november': 11, 'december': 12
  }

  query_lower = query.lower()
  for month_name, month_num in months.items():
    if month_name in query_lower:
      filter_dict['month'] = month_num
      break

  return filter_dict if filter_dict else None

**7.3 Enhanced retriever with metadata filtering**

In [ ]:
def get_filtered_retriever(query, k=5):
  """
  Create a retriever with optional metadata filtering based on the query.
  Automatically detects date filters in the query.

  Args:
    query: User's query string
    k: Number of documents to retrieve (default: 5)

  Returns:
    Configured retriever that may include metadata filters
  """
  # Parse query for date filters
  metadata_filter = parse_date_filter(query)

  # Build search kwargs
  search_kwargs = {"k": k}

  # Add metadata filter if found (using Chroma's filter syntax)
  if metadata_filter:
    # Convert to Chroma's $and/$eq format
    filter_conditions = []
    for key, value in metadata_filter.items():
      filter_conditions.append({key: {"$eq": value}})

    # Use $and if multiple conditions, otherwise single condition
    if len(filter_conditions) > 1:
      search_kwargs["filter"] = {"$and": filter_conditions}
    else:
      search_kwargs["filter"] = filter_conditions[0]

    print(f" Applying metadata filter: {metadata_filter}")

  # Create and return retriever
  return vector_store.as_retriever(
    search_type="similarity",
    search_kwargs=search_kwargs
      )

**7.4 Test retrieval system**

In [ ]:
# Test 1: Basic retrieval without date filter
test_query_1 = "What is video assisted refereeing in football?"

print(f"Query: {test_query_1}")
print("-" * 60)

test_retriever_1 = get_filtered_retriever(test_query_1, k=3)
test_docs_1 = test_retriever_1.invoke(test_query_1)

print(f"\n Retrieved {len(test_docs_1)} documents")
print(f" Article titles: {related_titles(test_docs_1)}")

# Show first document preview
if test_docs_1:
  print(f"\n First document preview:")
  print(f"   Title: {test_docs_1[0].metadata.get('title', 'Unknown')}")
  print(f"   Date: {test_docs_1[0].metadata.get('year', '')}-{test_docs_1[0].metadata.get('month', '')}-{test_docs_1[0].metadata.get('day', '')}")
  print(f"   Content preview: {test_docs_1[0].page_content[:200]}...")

Query: What is video assisted refereeing in football?
------------------------------------------------------------

 Retrieved 3 documents
 Article titles: ["« Nous avons réussi à réduire la place des polémiques sur l'arbitrage »", "L'excès de ralentis nuit gravement à la vérité"]

 First document preview:
   Title: « Nous avons réussi à réduire la place des polémiques sur l'arbitrage »
   Date: 2010-10-
   Content preview: Nous ne savions pas qu'Olivier Rouyer était contre l'arbitrage vidéo et pourtant, nous l'écoutons très régulièrement...
Il fait même partie des consultants les plus obsédés par l'arbitrage, à notre se...


In [ ]:
# Test 2: Retrieval with date filter
test_query_2 = "What topics were discussed in articles published in March 2016?"

print(f"\n\nQuery: {test_query_2}")
print("-" * 60)

test_retriever_2 = get_filtered_retriever(test_query_2, k=3)
test_docs_2 = test_retriever_2.invoke(test_query_2)

print(f"\n Retrieved {len(test_docs_2)} documents")
print(f" Article titles: {related_titles(test_docs_2)}")

# Show metadata for all retrieved docs to verify filtering
print(f"\n Document dates:")
for i, doc in enumerate(test_docs_2, 1):
  year = doc.metadata.get('year', 'N/A')
  month = doc.metadata.get('month', 'N/A')
  print(f"   Doc {i}: {year}-{month:02d}" if isinstance(month, int) else f"   Doc {i}: {year}-{month}")



Query: What topics were discussed in articles published in March 2016?
------------------------------------------------------------
 Applying metadata filter: {'year': 2016, 'month': 3}


InternalError: Error executing plan: Internal error: Error finding id

In [ ]:
# Test 3: Document formatting for prompts
print(f"\n\n Testing document formatting:")
print("-" * 60)

formatted = format_docs_with_metadata(test_docs_1[:2])  # Format first 2 docs
print(formatted[:500] + "..." if len(formatted) > 500 else formatted)

**8.Load chat model**

In [ ]:
# Test Connection
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_base=LLM_LITE_API_BASE,
    api_key=LLM_LITE_API_KEY,
    model=LLM_MODEL,
    temperature=0.1
)

answer = llm.invoke(
    [{"role": "user", "content": "Y a-t-il du beurre dans un croissant ?"}]
)

**9.Define RAG answer function**

In [ ]:
def rag_answer(query):

    filtered_retriever = get_filtered_retriever(query, k=3)
    docs = filtered_retriever.invoke(query)

    if not docs:
        return {
            "answer": "I don't know",
            "titles": []
        }

    titles = related_titles(docs)
    context = format_docs_with_metadata(docs)

    prompt = f"""
The following documents come from football articles (originally written in French).

Answer the question in English using only the information in the documents.
If the answer is not in the documents, reply exactly: I don't know

Documents:
{context}

Question: {query}

Answer:
"""

    response = llm.invoke([{"role": "user", "content": prompt}])
    answer = response.content.strip()

    return {
        "answer": answer,
        "titles": titles
    }

**10. Test the RAG system**

In [ ]:
query = "What is video assisted refereeing in football?"

result = rag_answer(query)

print("Query:", query)
print("\nAnswer:")
print(result["answer"])

print("\nRelated articles:")
for t in result["titles"]:
    print("-", t)

**11. Test unrelated query**

In [ ]:
query = "Explain the concept of manifest destiny"

result = rag_answer(query)

print("\nQuery:", query)
print("\nAnswer:")
print(result["answer"])

In [ ]:
vector_store._collection.count()